# 03 Two-stage model (`two_stage_ffn_rf`): feed-forward network for the level, random forest for the flat: training and inference

The static + temporal network of notebook 02 multiplies the temporal and static inputs inside its hidden layers, so errors in how the market moves are carried into every flat. This notebook separates the two:
**the time effect and the flat effect are fitted by different models and combined without interaction.**

* **Stage 1 (level by month):** the temporal-features network of notebook 01 (`177 -> 64 -> 16 -> 1`, LeakyReLU), which predicts the rent level of each month from the temporal features (`feature_config_temporal.py`).
* **Stage 2 (the flat):** a random forest on the static features (the 151 of notebook 02) that predicts the **ratio** `rent / level of its month`. Because rents rose by about 50% over the period, a flat's premium over the average flat grows with the level,
  so the ratio is the right target (not the dollar difference). The fit is weighted by `level^2`, which makes the forest's squared error in ratio the squared error in dollars.
* **Prediction:** `level of the month x ratio of the flat`. The forest never sees the month, so the time effect cannot interact with the flat. The model is multiplicative, so **the growth of the same flats equals the growth of the stage-1 level by construction**:
  stage 2 cannot move the trend, and a bad trend passes straight through.

Model selection on the held-out fit (2021-01..2024-03, validated on 2024-04..2025-03):

1. **Forest settings:** a small grid over the share of features per split and the minimum leaf size, scored with the true monthly level (a perfect stage 1) so that only stage 2 is judged.
2. **Stage-1 inputs:** the network on all 177 temporal features against networks on the 6 macro features and on macro + COE (18), over 5 seeds each, since a network's extrapolation depends on the seed. The choice is checked on the level of the
   validation months, on the two-stage error of the validation rows, and on the growth the refitted network forecasts for 2026. The settings are picked on the validation rows, so the held-out predictions are slightly optimistic.

Then the **final fit** (2021-01..2025-03) uses the stage-1 network of notebook 01 (run it first) with the forest refitted on all the rows.

* **Outputs:** the forest in `results/models/`, the model-selection tables in `results/tables/two_stage_ffn_rf/`, the predictions and the fixed-flat levels in `03_evaluation/predictions/`. Evaluation: `03_evaluation/evaluation.ipynb`.

In [1]:
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

sys.path.insert(0, str(Path.cwd().parents[1]))     # 02_modelling: nn_utils.py is shared by the single-stage and two-stage notebooks
import nn_utils as nu

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
NAME = 'two_stage_ffn_rf'
SEEDS = range(5)
nu.use_results_dir(Path.cwd() / 'results')
nu.ensure_output_dirs(NAME)

## 1. Data

In [2]:
train, test, STATIC = nu.load_static()
fit, val = (train['sample_type'] == 'train').to_numpy(), (train['sample_type'] == 'val').to_numpy()
month, y = train['month'].to_numpy(), train[nu.LABEL].to_numpy()
X, X_test = train[STATIC].to_numpy(np.float32), test[STATIC].to_numpy(np.float32)
FIT_MONTHS, VALIDATION_MONTHS = np.arange(nu.TRAIN_LAST + 1), np.arange(nu.TRAIN_LAST + 1, nu.VALIDATION_LAST + 1)
month_mean = train.groupby('month')[nu.LABEL].mean()                                 # the actual level of each month 0..50
y_val, m_val = y[val], month[val]

temporal = nu.load_temporal()
ALL = list(temporal.columns)
MACRO = list(nu.MACRO_FEATURE)
MACRO_COE = MACRO + list(nu.COE_FEATURE)
ALL_NAME = f'network, all ({len(ALL)})'
STAGE1 = {'network, macro (6 features)': MACRO, 'network, macro + COE (18)': MACRO_COE, ALL_NAME: ALL}
print(f'static {len(STATIC)} features; fit rows {fit.sum():,} (months 0..{nu.TRAIN_LAST}), validation rows {val.sum():,} (months {nu.TRAIN_LAST + 1}..{nu.VALIDATION_LAST}), test rows {len(test):,}')
print({k: len(v) for k, v in STAGE1.items()})

static 151 features; fit rows 114,841 (months 0..38), validation rows 35,158 (months 39..50), test rows 50,000
{'network, macro (6 features)': 6, 'network, macro + COE (18)': 18, 'network, all (177)': 177}


## 2. Held-out fit of stage 1

Each network is fitted on the rows of months 0..38, for 5 seeds, and asked for the level of every month. The network on all 177 features (seed 0) is the stage-1 model of the held-out two-stage fit.

In [3]:
held = {name: {} for name in STAGE1}
for name, features in STAGE1.items():
    Z = temporal[features]
    for seed in SEEDS:
        held[name][seed] = nu.month_levels(nu.fit_ffn(Z, month[fit], y[fit], seed=seed), Z)
level_in = held[ALL_NAME][0]                                                          # stage-1 level, fitted on months 0..38
level_w = level_in.loc[month[fit]].to_numpy()

## 3. Stage 2: forest settings

The forest is fitted once, on the rows of months 0..38: target = rent / level of its month (the in-sample level of the seed-0 network above), weighted by `level^2`. (The in-sample level of every stage-1 choice is within a few dollars of the month's
actual mean rent, so the same forest serves all of them.) Grid: the share of features tried per split and the minimum leaf size (200 trees, depth 24, 60% of the rows per tree), scored on the validation rows with the **oracle level**, the true monthly mean of each
validation month, i.e. a perfect stage 1. The earlier pipeline's settings (128 of its 237 features per split, leaf 10) correspond to 0.85 and 10 here.

In [4]:
oracle = month_mean.loc[m_val].to_numpy()
BASE_PARAMS = {'n_estimators': 200, 'max_depth': 24, 'max_samples': 0.6}
grid = []
for max_features in (0.33, 0.55, 0.85):
    for min_samples_leaf in (5, 10, 20):
        forest = RandomForestRegressor(n_jobs=-1, random_state=0, max_features=max_features, min_samples_leaf=min_samples_leaf, **BASE_PARAMS
                                       ).fit(X[fit], y[fit] / level_w, sample_weight=level_w ** 2)
        grid.append({'max_features': max_features, 'min_samples_leaf': min_samples_leaf, 'validation_rmse_oracle_level': nu.rmse(y_val, oracle * forest.predict(X[val]))})
grid = pd.DataFrame(grid).sort_values('validation_rmse_oracle_level').reset_index(drop=True)
print(grid.round(2).to_string())
RF_PARAMS = {**BASE_PARAMS, 'max_features': float(grid.loc[0, 'max_features']), 'min_samples_leaf': int(grid.loc[0, 'min_samples_leaf'])}
print('\nforest parameters:', RF_PARAMS)

   max_features  min_samples_leaf  validation_rmse_oracle_level
0          0.33                10                        469.13
1          0.55                10                        469.25
2          0.55                20                        469.50
3          0.85                20                        469.51
4          0.85                10                        469.51
5          0.33                20                        469.72
6          0.33                 5                        470.03
7          0.55                 5                        470.54
8          0.85                 5                        470.96

forest parameters: {'n_estimators': 200, 'max_depth': 24, 'max_samples': 0.6, 'max_features': 0.33, 'min_samples_leaf': 10}


## 4. Held-out two-stage fit, and the choice of the stage-1 inputs

The forest with the chosen settings, fitted on months 0..38, gives the held-out predictions: stage-1 level (seed 0, all 177 features) x forest ratio. The same forest is combined with every stage-1 choice to compare them
(`stage1_selection.csv`): the error of the level of months 39..50, the two-stage RMSE on the validation rows, and the growth at mid-2026 (2026-06 over 2025-01..03) of the same network refitted on months 0..50 (5 seeds).
"carry-forward" repeats the actual level of month 38; "straight-line" is a linear trend of the monthly mean rent of months 0..38.

In [5]:
forest_held = RandomForestRegressor(n_jobs=-1, random_state=0, **RF_PARAMS).fit(X[fit], y[fit] / level_w, sample_weight=level_w ** 2)
ratio_val = forest_held.predict(X[val])
pred_val = level_in.loc[m_val].to_numpy() * ratio_val

two_stage_rmse = lambda levels: nu.rmse(y_val, levels.loc[m_val].to_numpy() * ratio_val)
growth = lambda lv: lv.loc[65] / lv.loc[nu.Q1_2025].mean() - 1                       # 2026-06 over 2025-01..03
def level_scores(levels):
    err = levels.loc[VALIDATION_MONTHS] - month_mean.loc[VALIDATION_MONTHS]
    return {'level_rmse': float(np.sqrt((err ** 2).mean())), 'level_bias': float(err.mean()), 'last_month_error': float(err.iloc[-1])}

slope, intercept = np.polyfit(FIT_MONTHS, month_mean.loc[FIT_MONTHS], 1)
carry, trend = pd.Series(month_mean.loc[nu.TRAIN_LAST], index=range(nu.N_MONTHS)), pd.Series(intercept + slope * np.arange(nu.N_MONTHS))
selection = {'oracle level (true monthly mean)': {'two_stage_rmse': two_stage_rmse(month_mean)},
             'carry-forward (month 38)': {**level_scores(carry), 'two_stage_rmse': two_stage_rmse(carry)},
             'straight-line trend': {**level_scores(trend), 'two_stage_rmse': two_stage_rmse(trend)}}
for name, features in STAGE1.items():
    Z = temporal[features]
    g = np.array([growth(nu.month_levels(nu.fit_ffn(Z, month, y, seed=s), Z)) for s in SEEDS])         # refitted on months 0..50
    scores, rmses = pd.DataFrame([level_scores(held[name][s]) for s in SEEDS]), np.array([two_stage_rmse(held[name][s]) for s in SEEDS])
    selection[name] = {**scores.mean().to_dict(), 'level_rmse_sd': scores['level_rmse'].std(), 'two_stage_rmse': rmses.mean(), 'two_stage_rmse_sd': rmses.std(ddof=1),
                       'growth_mid_2026_mean': g.mean(), 'growth_mid_2026_min': g.min(), 'growth_mid_2026_max': g.max()}
selection = pd.DataFrame(selection).T
print(selection.round(3).to_string())

                                  two_stage_rmse  level_rmse  level_bias  last_month_error  level_rmse_sd  two_stage_rmse_sd  growth_mid_2026_mean  growth_mid_2026_min  growth_mid_2026_max
oracle level (true monthly mean)         469.134         NaN         NaN               NaN            NaN                NaN                   NaN                  NaN                  NaN
carry-forward (month 38)                 470.104      38.403     -28.990           -65.902            NaN                NaN                   NaN                  NaN                  NaN
straight-line trend                      613.394     383.662     368.448           523.506            NaN                NaN                   NaN                  NaN                  NaN
network, macro (6 features)              539.406     264.891    -225.387          -460.241         69.092             35.929                -0.113               -0.132               -0.068
network, macro + COE (18)                500.633     17

## 5. Final fit: 2021-01..2025-03

Stage 1 is the saved network of notebook 01 (months 0..50, seed 0; all 177 temporal features), unchanged: its predicted level for the 67 months. Stage 2 is refitted on all the train and validation rows with the ratio `rent / level of its month`.
The level of the same flats in each month is level(month) x the mean ratio of the fixed flats (the train flats approved in 2025-01..03).

In [6]:
net1, ckpt = nu.load_ffn(nu.MODEL_DIR / 'temporal_ffn.pt')
Z1 = temporal[ckpt['temporal_features']]
stage1 = nu.month_levels(net1, Z1)                                                   # stage 1: months 0..66
assert list(stage1.index) == list(range(nu.N_MONTHS))

level_w = stage1.loc[month].to_numpy()
forest = RandomForestRegressor(n_jobs=-1, random_state=0, **RF_PARAMS).fit(X, y / level_w, sample_weight=level_w ** 2)
ratio_tv, ratio_test = forest.predict(X), forest.predict(X_test)
pred_train, pred_test = stage1.loc[month].to_numpy() * ratio_tv, stage1.loc[test['month']].to_numpy() * ratio_test
level = stage1 * ratio_tv[np.isin(month, nu.Q1_2025)].mean()                         # the same flats in every month: level(m) x mean ratio of the flats
print(f'stage 2 fitted on {len(y):,} rows; in-sample ratio mean {ratio_tv.mean():.3f}; test ratio mean {ratio_test.mean():.3f}')
print(level.loc[44:].round(0).rename(lambda m: nu.month_label(m)[0]).to_frame('level of the same flats').T.to_string())

stage 2 fitted on 149,999 rows; in-sample ratio mean 1.005; test ratio mean 1.007
                         2024-09  2024-10  2024-11  2024-12  2025-01  2025-02  2025-03  2025-04  2025-05  2025-06  2025-07  2025-08  2025-09  2025-10  2025-11  2025-12  2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  2026-07
level of the same flats   3106.0   3111.0   3134.0   3117.0   3121.0   3131.0   3156.0   3120.0   3198.0   3191.0   3232.0   3195.0   3183.0   3223.0   3193.0   3193.0   3202.0   3223.0   3202.0   3270.0   3293.0   3340.0   3344.0


## 6. Save the model, the model-selection tables and the predictions

In [7]:
paths = [nu.MODEL_DIR / f'{NAME}_forest.joblib']
joblib.dump({'forest': forest, 'static_features': STATIC, 'rf_params': RF_PARAMS, 'target': 'rent / stage-1 level of the month, weighted by level^2', 'stage1_model': 'models/temporal_ffn.pt'}, paths[0], compress=3)
paths += [nu.save_table(grid, NAME, 'rf_tuning.csv', index=False), nu.save_table(selection, NAME, 'stage1_selection.csv', index_label='stage 1')]
paths += [nu.save_predictions(NAME, 'val', train[val], pred_val),        # held-out: stage 1 and forest fitted on 2021-01..2024-03
          nu.save_predictions(NAME, 'train', train, pred_train),         # in-sample: the final model, all 149,999 train + validation rows
          nu.save_predictions(NAME, 'test', test, pred_test),            # the final model, the 50,000 test rows
          nu.save_levels(NAME, level)]
print('saved:', *paths, sep='\n  ')

saved:
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/neural_network/results/models/two_stage_ffn_rf_forest.joblib
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/neural_network/results/tables/two_stage_ffn_rf/rf_tuning.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/02_modelling/two_stage_models/neural_network/results/tables/two_stage_ffn_rf/stage1_selection.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/val/two_stage_ffn_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/train/two_stage_ffn_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/test/two_stage_ffn_rf.csv
  /Users/keltonn/Desktop/Repo/school/CS5228-project/final_report/03_evaluation/predictions/levels/two_stage_ffn_rf.csv
